In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import requests
from pandas import json_normalize
import time


from sqlalchemy import create_engine
import psycopg2
from psycopg2.extras import execute_values


import json
import os


In [2]:
with open('config.json') as config_file:
    config = json.load(config_file)

In [7]:
conn = psycopg2.connect(host = config['db_host'],
                        dbname = config['db_name'] ,
                        user = config['db_user'],
                        password = config['db_password'] ,
                        port = config['db_port'])

In [4]:
API_KEY = config['api_key']

Can add more below as needed and rerun this script.

In [6]:
desired_variables = ['INFLATION','UNEMPLOYMENT','BRENT','FEDERAL_FUNDS_RATE','COFFEE']

if data[interval] is annual then use below logic, else just pull months in

In [7]:
all_data = []  # To store data for all variables

for var in desired_variables:
    url = f'https://www.alphavantage.co/query?function={var}&apikey={API_KEY}'
    response = requests.get(url)
    data = response.json()

    if 'data' not in data:  # Handle API errors
        print(f"Error fetching {var}: {data.get('error', 'Unknown error')}")
        continue

    data_list = data['data']
    name = var
    unit = data.get('unit', '')

    if data.get('interval') == 'annual':  
        expanded_data = []
        for entry in data_list:
            year = entry['date'][:4]
            value = entry['value']
            for month in range(1, 13):
                date = f"{year}-{month:02d}-01"
                expanded_data.append({'name': name, 'unit': unit, 'date': date, 'value': value})
        all_data.extend(expanded_data)
    else:  
        for entry in data_list:
            all_data.append({'name': name, 'unit': unit, 'date': entry['date'], 'value': entry['value']})

# Convert to DataFrame
df = pd.DataFrame(all_data)
df['value'] = pd.to_numeric(df['value'], errors='coerce')  # Ensure numeric values

In [12]:
df.rename(columns={
    'name': 'indicator_name',
    'unit': 'unit_of_measure',
    'date': 'data',
    'value': 'indicator_value'
}, inplace=True)

In [13]:
uniques = df[['indicator_name','unit_of_measure']].drop_duplicates()

In [14]:
uniques

,indicator_name,unit_of_measure
0,INFLATION,percent
768,UNEMPLOYMENT,percent
1695,BRENT,dollars per barrel
2150,FEDERAL_FUNDS_RATE,percent
2999,COFFEE,cents per pound


In [15]:
# Assuming you already have your DataFrame 'uniques' loaded and have established the connection
data_to_insert = uniques[['indicator_name', 'unit_of_measure']].values.tolist()

# Insert data, skipping duplicates based on 'indicator_name'
insert_query = """
    INSERT INTO economic_indicators (indicator_name, unit_of_measure)
    VALUES %s
    ON CONFLICT (indicator_name) DO NOTHING;
"""

cursor = conn.cursor()

# Use execute_values to insert multiple rows at once
execute_values(cursor, insert_query, data_to_insert)

# Commit the transaction to save changes
conn.commit()

# Print success message
print("Data inserted successfully.")

# Close the cursor and connection
cursor.close()
conn.close()

Data inserted successfully.


***

Now that the initial economic variables are set up, we will pull from the table to get indicator_id for each indicator_name and build out the dataframe with indicator_id,data_date,indicator_value, and data_source as 'AlphaVantage' for now.

In [16]:
conn = psycopg2.connect(
    host=config['db_host'],
    dbname=config['db_name'],
    user=config['db_user'],
    password=config['db_password'],
    port=config['db_port']
)

In [17]:
def check_available():
    cursor = conn.cursor()
    cursor.execute("SELECT indicator_id,indicator_name FROM economic_indicators")
    indicators = pd.DataFrame(cursor.fetchall(), columns=["indicator_id","indicator_name"])

    
    return(indicators)

In [18]:
ca = check_available()

In [19]:
ca

,indicator_id,indicator_name
0,1,INFLATION
1,2,UNEMPLOYMENT
2,3,BRENT
3,4,FEDERAL_FUNDS_RATE
4,5,COFFEE


Extract based on indicator_names present in above df

In [30]:
# Function to fetch and process data
def extract(row):
    indicator_id = row['indicator_id']
    indicator_name = row['indicator_name']

    url = f'https://www.alphavantage.co/query?function={indicator_name}&apikey={API_KEY}'
    response = requests.get(url)
    data = response.json()

    if 'data' not in data:  # Handle API errors
        print(f"Error fetching {indicator_name}: {data.get('error', 'Unknown error')}")
        return pd.DataFrame()  # Return an empty DataFrame if API fails

    all_data = []

    if data.get('interval') == 'annual':  
        for entry in data['data']:
            year = entry['date'][:4]
            value = entry['value']
            for month in range(1, 13):
                date = f"{year}-{month:02d}-01"
                all_data.append({'indicator_id': indicator_id,
                                 'data_date': date, 'indicator_value': value})
    else:  
        for entry in data['data']:
            all_data.append({'indicator_id': indicator_id,
                             'data_date': entry['date'], 'indicator_value': entry['value']})

    df = pd.DataFrame(all_data)
    df['indicator_value'] = pd.to_numeric(df['indicator_value'], errors='coerce')  # Ensure numeric values
    df['data_source'] = 'Alpha'
    return df
  

In [31]:
extract_res = []

for _, row in ca.iterrows():
    result = extract(row)
    extract_res.append(result)
    print(f"Fetched {row['indicator_name']}, sleeping to respect rate limit...")
    time.sleep(12)  # Sleep 12 seconds between requests

# Concatenate results into a single DataFrame
extract_res = pd.concat(extract_res, ignore_index=True)

Fetched INFLATION, sleeping to respect rate limit...
Fetched UNEMPLOYMENT, sleeping to respect rate limit...
Fetched BRENT, sleeping to respect rate limit...
Fetched FEDERAL_FUNDS_RATE, sleeping to respect rate limit...
Fetched COFFEE, sleeping to respect rate limit...


In [32]:
extract_res

,indicator_id,data_date,indicator_value,data_source
0,1,2023-01-01,4.116338,Alpha
1,1,2023-02-01,4.116338,Alpha
2,1,2023-03-01,4.116338,Alpha
3,1,2023-04-01,4.116338,Alpha
4,1,2023-05-01,4.116338,Alpha
...,...,...,...,...
3536,5,1980-05-01,NaN,Alpha
3537,5,1980-04-01,NaN,Alpha
3538,5,1980-03-01,NaN,Alpha
3539,5,1980-02-01,NaN,Alpha


In [33]:
extract_res.loc[extract_res['indicator_id']==5]

,indicator_id,data_date,indicator_value,data_source
2999,5,2025-02-01,409.516500,Alpha
3000,5,2025-01-01,353.933478,Alpha
3001,5,2024-12-01,344.118636,Alpha
3002,5,2024-11-01,304.952857,Alpha
3003,5,2024-10-01,276.777391,Alpha
...,...,...,...,...
3536,5,1980-05-01,NaN,Alpha
3537,5,1980-04-01,NaN,Alpha
3538,5,1980-03-01,NaN,Alpha
3539,5,1980-02-01,NaN,Alpha


- how to deal with api max call for 1 min
- transform not really needed
- load to table and set up airflow process

In [38]:
# Prepare insert query template
cur = conn.cursor()


insert_query = """
    INSERT INTO indicator_data (indicator_id, data_date, indicator_value, data_source)
    VALUES %s
    ON CONFLICT (indicator_id, data_date, data_source) 
    DO NOTHING;
"""

# Prepare data as a list of tuples
data_tuples = list(extract_res.itertuples(index=False, name=None))

# Execute in one batch
execute_values(cur, insert_query, data_tuples)

# Finalize
conn.commit()
cur.close()
conn.close()

print("Fast batch insert complete.")

Fast batch insert complete.
